# 讓 Agent 判斷什麼時候要叫工具這份教材的主題是工具呼叫的交接邊界：工具規格給模型看，真正執行工具的是你的應用層。

## 在 Colab 準備環境先準備 SDK 執行環境。

In [ ]:
from pathlib import Pathimport os, sys, subprocess, jsonif not Path('agentic_sdk').exists():    if not Path('Agentic-SDK').exists():        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)    os.chdir('Agentic-SDK')subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)print('Agentic SDK ready')

## 先定義工具可以接收什麼這份 schema 是給模型看的。它描述工具名稱、用途，以及工具需要哪些欄位。這是本章最需要拆開看的設定。

In [ ]:
tools = [    {        'type': 'function',        'function': {            'name': 'create_support_ticket',            'description': '建立客服處理單。',            'parameters': {                'type': 'object',                'properties': {                    'title': {'type': 'string', 'description': '處理單標題'},                    'priority': {'type': 'string', 'description': '優先度'},                },                'required': ['title', 'priority'],                'additionalProperties': False,            },        },    }]tools

## 再準備真正會做事的函式SDK 產生工具呼叫，但不直接執行外部工具。真正做事的函式通常在你的應用層或後端服務裡。

In [ ]:
def create_support_ticket(title, priority):    if priority not in {'low', 'normal', 'high'}:        raise ValueError('priority 必須是 low、normal 或 high')    return {'ticket_id': 'TCK-1001', 'title': title, 'priority': priority, 'status': 'created'}

## 先看工具呼叫結果長什麼樣正式使用時，這份結果會由 `ToolCallAction` 產生。MVP 教材先用固定結果，讓你專心看應用層怎麼接手。

In [ ]:
latest_tool_calls = [    {        'id': 'call_demo_001',        'type': 'function',        'function': {            'name': 'create_support_ticket',            'arguments': json.dumps({'title': '無法登入 AI Hub', 'priority': 'high'}, ensure_ascii=False),        },    }]latest_tool_calls

## 解析並檢查工具參數模型給的參數不能直接相信。應用層要先解析、檢查，再決定是否執行。

In [ ]:
call = latest_tool_calls[0]function = call['function']arguments = json.loads(function['arguments'])print(function['name'])print(arguments)

## 由應用層執行工具通過檢查後，才呼叫真正的函式。這一步可以接資料庫、後端 API 或其他系統。

In [ ]:
if function['name'] == 'create_support_ticket':    tool_result = create_support_ticket(**arguments)    print(tool_result)

## 接上正式模型時正式使用時，把固定的 `latest_tool_calls` 換成 `ToolCallAction` 的輸出即可。請記得：權限控管、參數驗證、錯誤處理都在應用層。

In [ ]:
from agentic_sdk.modules import ToolCallAction# 需要 OpenAI-compatible 模型端點時，再建立 ToolCallAction。# action = ToolCallAction(#     api_key='<API key>',#     base_url='<OpenAI-compatible base_url>',#     model='<支援工具呼叫的模型>',#     tools=tools,# )